In [22]:
# Question 1. Converting PDFs to Markdown

In [23]:
## How many lines are in the extracted content from the "Think Python" book (2nd edition)?

In [13]:
import csv
import urllib.request
from pathlib import Path
from openai import OpenAI

openai_client = OpenAI()


CSV_FILE = "books.csv"
OUTPUT_DIR = Path("books")


def download(url, dest):
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as response, open(dest, "wb") as f:
        while chunk := response.read(1024 * 1024):
            f.write(chunk)


def main():
    OUTPUT_DIR.mkdir(exist_ok=True)

    with open(CSV_FILE, newline="") as f:
        books = list(csv.DictReader(f))

    for book in books:
        url = book["pdf_url"]
        dest = OUTPUT_DIR / url.split("/")[-1]

        if dest.exists():
            print(f"Skipping {book['title']} (already downloaded)")
            continue

        print(f"Downloading {book['title']} -> {dest}")
        try:
            download(url, dest)
        except Exception as e:
            print(f"  Failed: {e}")
            dest.unlink(missing_ok=True)

    print("Done.")


if __name__ == "__main__":
    main()

Skipping Think Python 2e (already downloaded)
Skipping Think DSP (already downloaded)
Skipping Think Complexity 2e (already downloaded)
Skipping Think Java 2e (already downloaded)
Skipping Physical Modeling in MATLAB (already downloaded)
Skipping Think OS (already downloaded)
Skipping Think C++ (already downloaded)
Done.


In [2]:
!uv add 'markitdown[pdf]'


Resolved 138 packages in 2.61s                                       
Prepared 13 packages in 1.32s                                            
Uninstalled 1 package in 19ms
░░░░░░░░░░░░░░░░░░░░ [0/13] Installing wheels...                                warning: Failed to hardlink files; falling back to full copy. This may lead to degraded performance.
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 13 packages in 792ms                              
 ~ ai-engineering-buildcamp-codespaces==0.1.0 (from file:///workspaces/ai-engineering-buildcamp-codespaces)
 + click==8.5.0
 + cryptography==50.0.2
 + flatbuffers==25.12.19
 + magika==0.6.2
 + markdownify==1.2.3
 + markitdown==0.1.8
 + onnxruntime==1.30.0
 + pdfminer-six==20260107
 + pdfplumber==0.11.10
 + pillow==12.3.0
 + protobuf==7.36.2
 + pypdfium2==5.

In [3]:
from pathlib import Path

from markitdown import MarkItDown

INPUT_DIR = Path("books")
OUTPUT_DIR = Path("books_text")


def main():
    OUTPUT_DIR.mkdir(exist_ok=True)
    md = MarkItDown()

    for pdf in sorted(INPUT_DIR.glob("*.pdf")):
        dest = OUTPUT_DIR / f"{pdf.stem}.md"

        if dest.exists():
            print(f"Skipping {pdf.name} (already converted)")
            continue

        print(f"Converting {pdf.name} -> {dest}")
        result = md.convert(str(pdf))
        dest.write_text(result.text_content, encoding="utf-8")

    print("Done.")


if __name__ == "__main__":
    main()


2026-10-04 16:28:15.057726911 [W:onnxruntime:Default, device_discovery.cc:146 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


Converting PhysicalModelingInMatlab4.pdf -> books_text/PhysicalModelingInMatlab4.md
Converting Think-C.pdf -> books_text/Think-C.md
Converting thinkcomplexity2.pdf -> books_text/thinkcomplexity2.md
Converting thinkdsp.pdf -> books_text/thinkdsp.md
Converting thinkjava2.pdf -> books_text/thinkjava2.md
Converting thinkos.pdf -> books_text/thinkos.md
Converting thinkpython2.pdf -> books_text/thinkpython2.md
Done.


In [4]:
!wc -l books_text/thinkpython2.md

9702 books_text/thinkpython2.md


In [ ]:
## Question 2. Chunking for RAG

In [5]:
from pathlib import Path

from gitsource import chunk_documents

documents = []

for path in sorted(Path("books_text").glob("*.md")):
    lines = path.read_text(encoding="utf-8").splitlines()
    content = [line for line in lines if line.strip()]
    documents.append({"source": path.name, "content": content})

chunks = chunk_documents(documents, size=100, step=50)

python_chunks = [c for c in chunks if c["source"] == "thinkpython2.md"]
print(len(python_chunks))

189


In [ ]:
## Question 3. Indexing with minsearch

In [6]:
!uv add minsearch

Resolved 138 packages in 1ms
Checked 134 packages in 59ms


In [8]:
from minsearch import Index

from pathlib import Path

from gitsource import chunk_documents
from minsearch import Index

documents = []

for path in sorted(Path("books_text").glob("*.md")):
    lines = path.read_text(encoding="utf-8").splitlines()
    content = [line for line in lines if line.strip()]
    documents.append({"source": path.name, "content": content})

chunks = chunk_documents(documents, size=100, step=50)


def prepare_documents(chunks):
    prepared = []
    for chunk in chunks:
        prepared.append({
            "source": chunk["source"],
            "start": chunk["start"],
            "content": "\n".join(chunk["content"]),
        })
    return prepared


documents = prepare_documents(chunks)

index = Index(text_fields=["content"], keyword_fields=["source"])
index.fit(documents)

print(len(documents))

988


In [ ]:
##Question 4. Searching and RAG

In [9]:
results = index.search("python function definition", num_results=5)

In [10]:
results

[{'source': 'PhysicalModelingInMatlab4.md',
  'start': 1800,
  'content': "| ------ | ------------ | --- | --- |\n| total  | = total +    | a;  |     |\nend\nans = total\nIfyouwereusinganyofthosevariablenamesbeforecallingthisscript, youmightbesurprised\nto find, after running the script, that their values had changed. If you have two scripts that\n48 Functions\nuse the same variable names, you might find that they work separately and then break when\nyou try to combine them. This kind of interaction is called a collision.\nname\nAs the number of scripts you write increases, and they get longer and more complex, name\ncollisions become more of a problem. Avoiding this problem is one of several motivations for\nfunctions.\n| 5.2 | Defining | Functions |     |     |\n| --- | -------- | --------- | --- | --- |\nA function is like a script, except that each function has its own workspace, so any variables\ndefined inside a function only exist while the function is running and don’t interfer

In [ ]:
## Question 5 full rag
'''
Do RAG for "python function definition". What's the response?

You don't have to use OpenAI, you can use an alternative provider.

Next, change these functions to also return the number of input and output tokens.

How many input tokens did we use for this one RAG query?
'''

In [16]:
import json

instructions = """
You're a course assistant, your task is to answer the QUESTION from the
course students using the provided CONTEXT
"""

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    prompt = prompt_template.format(
        question=question,
        context=context
    ).strip()
    return prompt

def search(question):
    return index.search(question, num_results=5)

def llm(user_prompt, instructions, model='gpt-4o-mini'):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=messages
    )
    usage = {
        "input_tokens": response.usage.input_tokens,
        "output_tokens": response.usage.output_tokens,
    }

    return response.output_text, usage

def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer, usage = llm(prompt, instructions)
    return answer, usage


In [17]:
rag("python function definition")

("In Python, a function is defined using the `def` keyword followed by the function's name and parentheses containing any parameters the function accepts. Here's a general structure of a function definition:\n\n```python\ndef function_name(parameters):\n    # Code block for the function\n    return value\n```\n\n### Example:\nHere’s a simple function that takes two numbers as input and returns their sum:\n\n```python\ndef add_numbers(a, b):\n    result = a + b\n    return result\n```\n\n### Key Points:\n- The `def` keyword is required at the beginning of the function definition.\n- The function name follows (e.g., `add_numbers`), which should be descriptive.\n- Parameters (e.g., `a` and `b`) are specified in parentheses. These are the inputs to the function.\n- The code block that makes up the function is indented below the function definition.\n- The `return` statement is used to send a value back to the caller of the function.\n\nYou can call the function like this:\n\n```python\nsum

In [18]:
from pydantic import BaseModel, Field
from typing import Literal

class RAGResponse(BaseModel):
    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions")

In [ ]:
#question 6 structured vs unstructured output


In [19]:
def llm_structured(user_prompt, instructions, model='gpt-4o-mini'):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.parse(
        model=model,
        input=messages,
        text_format=RAGResponse
    )

    usage = {
        "input_tokens": response.usage.input_tokens,
        "output_tokens": response.usage.output_tokens,
    }

    return response.output_parsed, usage

def rag_structured(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer, usage = llm_structured(prompt, instructions)
    return answer, usage

In [20]:
rag_structured("python function definition")

(RAGResponse(answer='### Python Function Definition\n\nIn Python, a function is defined using the `def` keyword, followed by the function name and parentheses containing any input parameters. The body of the function is indented beneath the definition line. Below is a simple example of a function definition:\n\n```python\ndef my_function(x):\n    result = x * x  # Square the input\n    return result   # Return the calculated result\n```\n\n#### Explanation:  \n- **`def`**: This keyword signifies the start of a function definition.  \n- **`my_function`**: This is the name of the function. You can call this function by its name followed by parentheses.  \n- **`x`**: This is a parameter that the function takes as input. You can use any name you like.  \n- **`result = x * x`**: This is the code that runs when the function is called, performing the operation (squaring `x` in this case).  \n- **`return result`**: This statement allows the function to send a result back to the caller.\n\n### 